# Kiến trúc Model Transformer ZH↔VI

Notebook này giới thiệu từng khối của model dịch song ngữ Trung-Việt với:
- **Encoder-Decoder Transformer** (8 layers mỗi bên)
- **Grouped-Query Attention** + **RoPE**
- **SwiGLU FFN**
- **Contrastive Learning** cho sentence embedding


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional

# Config demo
vocab_size = 8000
d_model = 768
n_heads = 12
n_kv_heads = 4
d_ff = 3072
dropout = 0.01
rope_base = 10000.0
max_len = 32

batch_size = 4  # Demo batch
src_len = 16
tgt_len = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"\nCấu hình model:")
print(f"  Vocab size: {vocab_size:,}")
print(f"  d_model: {d_model}")
print(f"  d_ff: {d_ff}")
print(f"  Attention heads: {n_heads} (Q) / {n_kv_heads} (KV)")
print(f"  Max length: {max_len}")


Device: cpu

Cấu hình model:
  Vocab size: 8,000
  d_model: 768
  d_ff: 3072
  Attention heads: 12 (Q) / 4 (KV)
  Max length: 32


## 1. Token Embedding Layer

Chuyển token ID thành vector embedding, scale với √d_model


In [2]:
# Token Embedding
embedding = nn.Embedding(vocab_size, d_model, padding_idx=0).to(device)
emb_scale = math.sqrt(d_model)
emb_dropout = nn.Dropout(dropout)

# Demo input
src_ids = torch.randint(1, vocab_size, (batch_size, src_len)).to(device)
src_ids[:, -2:] = 0  # Padding

print("INPUT:")
print(f"  src_ids shape: {src_ids.shape}  # (batch, seq_len)")

# Forward
src_emb = embedding(src_ids) * emb_scale
src_emb = emb_dropout(src_emb)

print("\nOUTPUT:")
print(f"  src_emb shape: {src_emb.shape}  # (batch, seq_len, d_model)")
print(f"  Embedding scaled by: √{d_model} = {emb_scale:.2f}")


INPUT:
  src_ids shape: torch.Size([4, 16])  # (batch, seq_len)

OUTPUT:
  src_emb shape: torch.Size([4, 16, 768])  # (batch, seq_len, d_model)
  Embedding scaled by: √768 = 27.71


## 2. RMSNorm

Chuẩn hoá theo Root Mean Square, thay LayerNorm để tăng tốc


In [3]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(
            torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps
            )
        return self.weight * x / rms

# Demo
rms_norm = RMSNorm(d_model).to(device)

print("INPUT:")
print(f"  x shape: {src_emb.shape}")

normed = rms_norm(src_emb)

print("\nOUTPUT:")
print(f"  normed shape: {normed.shape}  # Same shape")
print(f"  RMS before: {src_emb.pow(2).mean(dim=-1).sqrt().mean().item():.4f}")
print(f"  RMS after: {normed.pow(2).mean(dim=-1).sqrt().mean().item():.4f}")


INPUT:
  x shape: torch.Size([4, 16, 768])

OUTPUT:
  normed shape: torch.Size([4, 16, 768])  # Same shape
  RMS before: 24.4223
  RMS after: 0.8750


## 3. RoPE (Rotary Position Embedding)

Mã hoá vị trí bằng cách xoay vector Q, K theo góc phụ thuộc vị trí


In [4]:
class RoPE(nn.Module):
    def __init__(self, d_model: int, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer("inv_freq", inv_freq)
        self._cos = None
        self._sin = None
        self._seq_len_cached = 0

    def _maybe_update_cache(self, seq_len: int, device, dtype):
        if seq_len > self._seq_len_cached or self._cos is None or self._cos.device != device:
            self._seq_len_cached = seq_len
            positions = torch.arange(seq_len, device=device, dtype=dtype)
            freqs = torch.outer(positions, self.inv_freq.to(device))
            self._cos = freqs.cos()
            self._sin = freqs.sin()

    def forward(self, x: torch.Tensor, seq_len: Optional[int] = None):
        seq_len = seq_len or x.size(-2)
        self._maybe_update_cache(seq_len, x.device, x.dtype)
        return self._cos[:seq_len], self._sin[:seq_len]

def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Áp dụng RoPE lên tensor (batch, heads, seq, d_k)"""
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    rot1 = x1 * cos - x2 * sin
    rot2 = x1 * sin + x2 * cos
    return torch.stack([rot1, rot2], dim=-1).flatten(-2)

# Demo
d_k = d_model // n_heads  # 64
rope = RoPE(d_k, base=rope_base).to(device)

# Giả sử Q sau linear projection
Q = torch.randn(batch_size, n_heads, src_len, d_k).to(device)

print("INPUT:")
print(f"  Q shape: {Q.shape}  # (batch, heads, seq, d_k)")

cos, sin = rope(Q, src_len)
Q_rope = apply_rope(Q, cos, sin)

print("\nOUTPUT:")
print(f"  cos shape: {cos.shape}  # (seq, d_k/2)")
print(f"  sin shape: {sin.shape}  # (seq, d_k/2)")
print(f"  Q_rope shape: {Q_rope.shape}  # Same as input")
print(f"  RoPE base: {rope_base}")


INPUT:
  Q shape: torch.Size([4, 12, 16, 64])  # (batch, heads, seq, d_k)

OUTPUT:
  cos shape: torch.Size([16, 32])  # (seq, d_k/2)
  sin shape: torch.Size([16, 32])  # (seq, d_k/2)
  Q_rope shape: torch.Size([4, 12, 16, 64])  # Same as input
  RoPE base: 10000.0


## 4. Grouped-Query Attention + RoPE

- **12 Query heads**, **4 KV heads** → mỗi KV head phục vụ 3 Q heads
- Áp dụng RoPE lên Q và K


In [5]:
class GroupedQueryAttentionRoPE(nn.Module):
    def __init__(self,
                 d_model: int, n_heads: int, n_kv_heads: int,
                 dropout: float, rope_base: float
                 ):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups = n_heads // n_kv_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)
        self.rope = RoPE(self.d_k, base=rope_base)

    def forward(self, q, k, v, key_padding_mask=None, attn_mask=None):
        B, T_q = q.size(0), q.size(1)
        T_k = k.size(1)
        Q = self.W_q(q).view(B, T_q, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, T_k, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, T_k, self.n_kv_heads, self.d_k).transpose(1, 2)

        # Apply RoPE
        cos_q, sin_q = self.rope(Q, T_q)
        cos_k, sin_k = self.rope(K, T_k)
        Q = apply_rope(Q, cos_q, sin_q)
        K = apply_rope(K, cos_k, sin_k)

        # Repeat KV heads
        K = K.repeat_interleave(self.n_groups, dim=1)
        V = V.repeat_interleave(self.n_groups, dim=1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if key_padding_mask is not None:
            scores = scores.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), float("-inf")
                )
        if attn_mask is not None:
            scores = scores.masked_fill(
                attn_mask.unsqueeze(0).unsqueeze(0), float("-inf")
                )
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T_q, -1)
        return self.W_o(out)

# Demo
gqa = GroupedQueryAttentionRoPE(d_model, n_heads, n_kv_heads, dropout, rope_base).to(device)

x = torch.randn(batch_size, src_len, d_model).to(device)
src_pad_mask = (src_ids == 0)

print("INPUT:")
print(f"  x shape: {x.shape}  # (batch, seq, d_model)")
print(f"  padding_mask shape: {src_pad_mask.shape}  # (batch, seq)")
print(f"\n  Config: {n_heads} Q heads, {n_kv_heads} KV heads")
print(f"  → {n_heads // n_kv_heads} Q heads per KV head")

attn_out = gqa(x, x, x, key_padding_mask=src_pad_mask)

print("\nOUTPUT:")
print(f"  attn_out shape: {attn_out.shape}  # (batch, seq, d_model)")


INPUT:
  x shape: torch.Size([4, 16, 768])  # (batch, seq, d_model)
  padding_mask shape: torch.Size([4, 16])  # (batch, seq)

  Config: 12 Q heads, 4 KV heads
  → 3 Q heads per KV head

OUTPUT:
  attn_out shape: torch.Size([4, 16, 768])  # (batch, seq, d_model)


## 5. FFN với SwiGLU

Feed-forward network với SwiGLU activation


In [6]:
class FFN_SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.d_ff = d_ff
        self.linear1 = nn.Linear(d_model, 2 * d_ff, bias=False)
        self.linear2 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.linear1(x)
        g, v = h[..., : self.d_ff], h[..., self.d_ff :]
        s = g * torch.sigmoid(g)  # SwiGLU
        out = self.linear2(s * v)
        return self.dropout(out)

# Demo
ffn = FFN_SwiGLU(d_model, d_ff, dropout).to(device)

x = torch.randn(batch_size, src_len, d_model).to(device)

print("INPUT:")
print(f"  x shape: {x.shape}  # (batch, seq, d_model={d_model})")
print(f"\n  FFN: {d_model} → {2*d_ff} → {d_ff} → {d_model}")

ffn_out = ffn(x)

print("\nOUTPUT:")
print(f"  ffn_out shape: {ffn_out.shape}  # (batch, seq, d_model)")


INPUT:
  x shape: torch.Size([4, 16, 768])  # (batch, seq, d_model=768)

  FFN: 768 → 6144 → 3072 → 768

OUTPUT:
  ffn_out shape: torch.Size([4, 16, 768])  # (batch, seq, d_model)


## 6. Encoder Layer

Một layer encoder: RMSNorm → Self-Attention → Residual → RMSNorm → FFN → Residual


In [7]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, dropout, rope_base):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = GroupedQueryAttentionRoPE(
            d_model, n_heads, n_kv_heads, dropout, rope_base
            )
        self.ln2 = RMSNorm(d_model)
        self.ffn = FFN_SwiGLU(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_pad_mask=None):
        x1 = self.ln1(x)
        attn = self.self_attn(x1, x1, x1, key_padding_mask=src_pad_mask)
        x = x + self.dropout(attn)
        x2 = self.ln2(x)
        return x + self.ffn(x2)

# Demo
enc_layer = EncoderLayer(d_model, n_heads, n_kv_heads, d_ff, dropout, rope_base).to(device)

x = torch.randn(batch_size, src_len, d_model).to(device)
src_pad_mask = (src_ids == 0)

print("INPUT:")
print(f"  x shape: {x.shape}")
print(f"  src_pad_mask shape: {src_pad_mask.shape}")

enc_out = enc_layer(x, src_pad_mask)

print("\nOUTPUT:")
print(f"  enc_out shape: {enc_out.shape}  # Same shape")
print(f"\n  → Stack 8 layers như này để tạo Encoder đầy đủ")


INPUT:
  x shape: torch.Size([4, 16, 768])
  src_pad_mask shape: torch.Size([4, 16])

OUTPUT:
  enc_out shape: torch.Size([4, 16, 768])  # Same shape

  → Stack 8 layers như này để tạo Encoder đầy đủ


## 7. Decoder Layer

Một layer decoder: Masked Self-Attn → Cross-Attn (với encoder) → FFN


In [8]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, dropout, rope_base):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = GroupedQueryAttentionRoPE(
            d_model, n_heads, n_kv_heads, dropout, rope_base)
        self.ln2 = RMSNorm(d_model)
        self.cross_attn = GroupedQueryAttentionRoPE(
            d_model, n_heads, n_kv_heads, dropout, rope_base)
        self.ln3 = RMSNorm(d_model)
        self.ffn = FFN_SwiGLU(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, y, enc_out, tgt_pad_mask=None, tgt_causal_mask=None, src_pad_mask=None):
        # Self-attention (masked)
        y1 = self.ln1(y)
        self_attn = self.self_attn(
            y1, y1, y1, key_padding_mask=tgt_pad_mask, attn_mask=tgt_causal_mask)
        y = y + self.dropout(self_attn)

        # Cross-attention
        y2 = self.ln2(y)
        cross_attn = self.cross_attn(y2, enc_out, enc_out, key_padding_mask=src_pad_mask)
        y = y + self.dropout(cross_attn)

        # FFN
        y3 = self.ln3(y)
        return y + self.ffn(y3)

# Demo
dec_layer = DecoderLayer(d_model, n_heads, n_kv_heads, d_ff, dropout, rope_base).to(device)

# Giả sử đã có encoder output và decoder input
enc_out = torch.randn(batch_size, src_len, d_model).to(device)
tgt_ids = torch.randint(1, vocab_size, (batch_size, tgt_len)).to(device)
tgt_emb = torch.randn(batch_size, tgt_len, d_model).to(device)

src_pad_mask = (src_ids == 0)
tgt_pad_mask = (tgt_ids == 0)
tgt_causal_mask = torch.triu(torch.ones(tgt_len, tgt_len, dtype=torch.bool), diagonal=1).to(device)

print("INPUT:")
print(f"  tgt_emb shape: {tgt_emb.shape}  # (batch, tgt_len, d_model)")
print(f"  enc_out shape: {enc_out.shape}  # (batch, src_len, d_model)")
print(f"  tgt_causal_mask shape: {tgt_causal_mask.shape}  # (tgt_len, tgt_len)")

dec_out = dec_layer(tgt_emb, enc_out, tgt_pad_mask, tgt_causal_mask, src_pad_mask)

print("\nOUTPUT:")
print(f"  dec_out shape: {dec_out.shape}  # (batch, tgt_len, d_model)")
print(f"\n  → Stack 8 layers như này để tạo Decoder đầy đủ")


INPUT:
  tgt_emb shape: torch.Size([4, 12, 768])  # (batch, tgt_len, d_model)
  enc_out shape: torch.Size([4, 16, 768])  # (batch, src_len, d_model)
  tgt_causal_mask shape: torch.Size([12, 12])  # (tgt_len, tgt_len)

OUTPUT:
  dec_out shape: torch.Size([4, 12, 768])  # (batch, tgt_len, d_model)

  → Stack 8 layers như này để tạo Decoder đầy đủ


## 8. Full Transformer Model

Kết hợp: Embedding → 8 Encoder layers → 8 Decoder layers → Output (tied weights)


In [9]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_kv_heads, d_ff, dropout,
                 rope_base, num_enc_layers=8, num_dec_layers=8):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, n_kv_heads, d_ff, dropout, rope_base)
            for _ in range(num_enc_layers)
        ])
        self.encoder_final_ln = RMSNorm(d_model)

        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, n_kv_heads, d_ff, dropout, rope_base)
            for _ in range(num_dec_layers)
        ])
        self.decoder_final_ln = RMSNorm(d_model)

        self.output_bias = nn.Parameter(torch.zeros(vocab_size))
        self.emb_scale = math.sqrt(d_model)

    def forward(self, src_ids: torch.Tensor, tgt_ids: torch.Tensor) -> torch.Tensor:
        # Masks
        src_pad = (src_ids == 0)
        tgt_pad = (tgt_ids == 0)
        tgt_in = tgt_ids[:, :-1]
        tgt_pad_in = tgt_pad[:, :-1]
        T = tgt_in.size(1)
        tgt_causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=src_ids.device), diagonal=1)

        # Encoder
        src_emb = self.emb_dropout(self.embedding(src_ids) * self.emb_scale)
        enc_out = src_emb
        for layer in self.encoder_layers:
            enc_out = layer(enc_out, src_pad)
        enc_out = self.encoder_final_ln(enc_out)

        # Decoder
        tgt_emb = self.emb_dropout(self.embedding(tgt_in) * self.emb_scale)
        dec_out = tgt_emb
        for layer in self.decoder_layers:
            dec_out = layer(dec_out, enc_out, tgt_pad_in, tgt_causal, src_pad)
        dec_out = self.decoder_final_ln(dec_out)

        # Output (tied embedding)
        return F.linear(dec_out, self.embedding.weight, self.output_bias)

# Demo
model = TransformerModel(vocab_size, d_model, n_heads, n_kv_heads, d_ff, dropout, rope_base).to(device)

src_ids = torch.randint(1, vocab_size, (batch_size, src_len)).to(device)
tgt_ids = torch.randint(1, vocab_size, (batch_size, tgt_len)).to(device)

print("INPUT:")
print(f"  src_ids shape: {src_ids.shape}  # (batch, src_len)")
print(f"  tgt_ids shape: {tgt_ids.shape}  # (batch, tgt_len)")
print(f"\nModel structure:")
print(f"  - Embedding: {vocab_size} → {d_model}")
print(f"  - Encoder: 8 layers")
print(f"  - Decoder: 8 layers")
print(f"  - Output: tied embedding weights")

logits = model(src_ids, tgt_ids)

print("\nOUTPUT:")
print(f"  logits shape: {logits.shape}  # (batch, tgt_len-1, vocab_size)")
print(f"  → Predict next token for each position")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")


INPUT:
  src_ids shape: torch.Size([4, 16])  # (batch, src_len)
  tgt_ids shape: torch.Size([4, 12])  # (batch, tgt_len)

Model structure:
  - Embedding: 8000 → 768
  - Encoder: 8 layers
  - Decoder: 8 layers
  - Output: tied embedding weights

OUTPUT:
  logits shape: torch.Size([4, 11, 8000])  # (batch, tgt_len-1, vocab_size)
  → Predict next token for each position

Total parameters: 157,179,200


## 9. Projection Head (Contrastive Learning)

Chiếu encoder output về không gian contrastive để học sentence embedding


In [10]:
class ProjectionHead(nn.Module):
    def __init__(self, input_dim: int, proj_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.ReLU(),
            nn.Linear(input_dim, proj_dim),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

def mean_pool(enc_out, ids, pad_id, special_ids):
    """Mean pooling, bỏ qua padding và special tokens"""
    mask = (ids != pad_id)
    for special in special_ids:
        mask = mask & (ids != special)
    mask = mask.float()
    summed = (enc_out * mask.unsqueeze(-1)).sum(dim=1)
    denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
    return summed / denom

# Demo
proj_dim = 768
projection = ProjectionHead(d_model, proj_dim).to(device)

# Giả sử có encoder output
enc_out = torch.randn(batch_size, src_len, d_model).to(device)
src_ids = torch.randint(1, vocab_size, (batch_size, src_len)).to(device)
pad_id = 0
special_ids = [0, 2, 3, 4, 5]  # pad, bos, eos, <2zh>, <2vi>

print("INPUT:")
print(f"  enc_out shape: {enc_out.shape}  # (batch, seq, d_model)")

# Mean pool
pooled = mean_pool(enc_out, src_ids, pad_id, special_ids)
print(f"\nAfter mean pooling:")
print(f"  pooled shape: {pooled.shape}  # (batch, d_model)")

# Project
z = projection(pooled)
print(f"\nAfter projection:")
print(f"  z shape: {z.shape}  # (batch, proj_dim)")
print(f"  z norm: {z.norm(dim=-1).mean().item():.4f}  # Should be ~1.0 (normalized)")

print(f"\n→ Dùng z để tính contrastive loss giữa các cặp câu ZH-VI")


INPUT:
  enc_out shape: torch.Size([4, 16, 768])  # (batch, seq, d_model)

After mean pooling:
  pooled shape: torch.Size([4, 768])  # (batch, d_model)

After projection:
  z shape: torch.Size([4, 768])  # (batch, proj_dim)
  z norm: 1.0000  # Should be ~1.0 (normalized)

→ Dùng z để tính contrastive loss giữa các cặp câu ZH-VI


## 10. Tổng quan kiến trúc

Sơ đồ luồng dữ liệu đầy đủ


In [11]:
print("=" * 70)
print("LUỒNG DỮ LIỆU TRANSFORMER ZH↔VI")
print("=" * 70)

print("\n1. INPUT")
print("   src_ids: (batch, src_len) → token IDs nguồn")
print("   tgt_ids: (batch, tgt_len) → token IDs đích")

print("\n2. ENCODER")
print("   ├─ Embedding(src_ids) * √768 → (batch, src_len, 768)")
print("   ├─ Dropout(0.01)")
print("   ├─ 8x EncoderLayer:")
print("   │   ├─ RMSNorm")
print("   │   ├─ Self-Attention (GQA: 12 Q heads / 4 KV heads + RoPE)")
print("   │   ├─ Residual")
print("   │   ├─ RMSNorm")
print("   │   ├─ FFN (SwiGLU: 768 → 6144 → 3072 → 768)")
print("   │   └─ Residual")
print("   └─ RMSNorm → enc_out: (batch, src_len, 768)")

print("\n3. DECODER")
print("   ├─ Embedding(tgt_ids[:-1]) * √768 → (batch, tgt_len-1, 768)")
print("   ├─ Dropout(0.01)")
print("   ├─ 8x DecoderLayer:")
print("   │   ├─ RMSNorm")
print("   │   ├─ Masked Self-Attention (GQA + RoPE + causal mask)")
print("   │   ├─ Residual")
print("   │   ├─ RMSNorm")
print("   │   ├─ Cross-Attention (Q từ decoder, K/V từ enc_out)")
print("   │   ├─ Residual")
print("   │   ├─ RMSNorm")
print("   │   ├─ FFN (SwiGLU)")
print("   │   └─ Residual")
print("   └─ RMSNorm → dec_out: (batch, tgt_len-1, 768)")

print("\n4. OUTPUT")
print("   └─ Linear(dec_out, embedding.weight.T) + bias")
print("      → logits: (batch, tgt_len-1, 8000)")

print("\n5. CONTRASTIVE (optional, chỉ dùng encoder)")
print("   ├─ Mean pool enc_out → (batch, 768)")
print("   ├─ ProjectionHead: Linear(768→768) → ReLU → Linear(768→768)")
print("   └─ L2 normalize → z: (batch, 768)")
print("      → Dùng z để tính contrastive loss giữa ZH-VI")

print("\n" + "=" * 70)
print("THÔNG SỐ QUAN TRỌNG")
print("=" * 70)
print(f"Vocab size:        {vocab_size:,}")
print(f"d_model:           {d_model}")
print(f"d_ff:              {d_ff}")
print(f"Attention heads:   {n_heads} Q heads / {n_kv_heads} KV heads")
print(f"Head dimension:    {d_model // n_heads}")
print(f"Encoder layers:    8")
print(f"Decoder layers:    8")
print(f"RoPE base:         {rope_base}")
print(f"Max length:        {max_len}")
print(f"Dropout:           {dropout}")
print("=" * 70)

print("\n✓ Notebook hoàn tất! Chạy từng cell để xem shape của từng khối.")


LUỒNG DỮ LIỆU TRANSFORMER ZH↔VI

1. INPUT
   src_ids: (batch, src_len) → token IDs nguồn
   tgt_ids: (batch, tgt_len) → token IDs đích

2. ENCODER
   ├─ Embedding(src_ids) * √768 → (batch, src_len, 768)
   ├─ Dropout(0.01)
   ├─ 8x EncoderLayer:
   │   ├─ RMSNorm
   │   ├─ Self-Attention (GQA: 12 Q heads / 4 KV heads + RoPE)
   │   ├─ Residual
   │   ├─ RMSNorm
   │   ├─ FFN (SwiGLU: 768 → 6144 → 3072 → 768)
   │   └─ Residual
   └─ RMSNorm → enc_out: (batch, src_len, 768)

3. DECODER
   ├─ Embedding(tgt_ids[:-1]) * √768 → (batch, tgt_len-1, 768)
   ├─ Dropout(0.01)
   ├─ 8x DecoderLayer:
   │   ├─ RMSNorm
   │   ├─ Masked Self-Attention (GQA + RoPE + causal mask)
   │   ├─ Residual
   │   ├─ RMSNorm
   │   ├─ Cross-Attention (Q từ decoder, K/V từ enc_out)
   │   ├─ Residual
   │   ├─ RMSNorm
   │   ├─ FFN (SwiGLU)
   │   └─ Residual
   └─ RMSNorm → dec_out: (batch, tgt_len-1, 768)

4. OUTPUT
   └─ Linear(dec_out, embedding.weight.T) + bias
      → logits: (batch, tgt_len-1, 8000)

5. 